# Geocoding — Forward & Reverse

Demonstrates the `geocoder()` function from the `airgap_geo` library, which
provides both **forward geocoding** (place name → coordinates via Nominatim) and
**reverse geocoding** (coordinates → address via Photon).

## Prerequisites

| Service                       | Default port | Role                                |
| ----------------------------- | ------------ | ----------------------------------- |
| Nominatim (forward geocoding) | 8080         | Place name / postcode → lat/lon     |
| Photon (reverse geocoding)    | 2322         | lat/lon → structured address fields |

See **Start Services** below for Docker commands.


______________________________________________________________________

## Start Services

Before running this notebook, make sure the required Docker services are up.
From the **project root**, run:

```bash
make geocoding-up        # starts Nominatim + Photon
```

Check running containers with `make ps`. See the main
[README](../README.md#docker-services) for all available profiles and
configuration details.


## Setup — Imports and Configuration


In [ ]:
import importlib
import pprint

import folium
import httpx
import pandas as pd
import requests

import airgap_geo.settings as _settings

importlib.reload(_settings)

from airgap_geo import geocoder
from airgap_geo.settings import NOMINATIM_URL, PHOTON_API

client = httpx.AsyncClient()

print("Configured service endpoints")
print(f"  Nominatim (forward geocoding) : {NOMINATIM_URL}")
print(f"  Photon    (reverse geocoding) : {PHOTON_API}")

## Health Check


In [ ]:
_SERVICES = {
    "Nominatim": NOMINATIM_URL,
    "Photon": PHOTON_API + "/api?q=London&limit=1",
}

rows = []
for name, base_url in _SERVICES.items():
    try:
        r = requests.get(base_url, timeout=5)
        rows.append(
            {
                "Service": name,
                "URL": base_url,
                "HTTP Status": r.status_code,
                "Reachable": "\u2713",
            }
        )
    except Exception:
        rows.append(
            {
                "Service": name,
                "URL": base_url,
                "HTTP Status": "ERR",
                "Reachable": "\u2717",
            }
        )

pd.DataFrame(rows).set_index("Service")

______________________________________________________________________

## 1 — Forward Geocoding (Nominatim)

`geocoder(location)` accepts a free-text place name or postcode. When the input is not a
valid `lat,lon` pair it forwards the query to **Nominatim** (`/search`), then enriches the
result with a reverse-geocode call to **Photon** to obtain a structured address.

The return value is a `GeocodeResult` with `.geo` (`GeoPoint`), `.address` (`Address`),
and `.raw` (the original Photon feature dict). Returns `None` on failure.


In [ ]:
# Forward geocode an address
result_address = await geocoder("10 Downing Street, London", client)
pprint.pprint(result_address)

In [ ]:
# A postcode string also resolves via Nominatim
result_postcode_geocode = await geocoder("EC2V 6DN", client)  # Bank of England
geo = result_postcode_geocode.geo if result_postcode_geocode else None
print(
    f"EC2V 6DN  \u2192  lat={geo.lat if geo else None}, lon={geo.lon if geo else None}"
)

In [ ]:
# Map the result - Nominatim result for 10 Downing Street
geo = result_address.geo if result_address else None
lat = geo.lat if geo else 51.5034
lon = geo.lon if geo else -0.1276

addr = result_address.address if result_address else None
label = (
    (addr.name if addr else None)
    or (addr.street if addr else None)
    or "10 Downing Street"
)

m_fwd = folium.Map(location=[lat, lon], zoom_start=16)
folium.Marker(
    [lat, lon],
    popup=folium.Popup(
        f"<b>{label}</b><br>lat={lat:.5f}, lon={lon:.5f}", max_width=250
    ),
    tooltip="Forward geocoding result",
    icon=folium.Icon(color="blue", icon="home"),
).add_to(m_fwd)
m_fwd

______________________________________________________________________

## 2 — Reverse Geocoding (Photon)

When the input to `geocoder()` is a valid `"lat,lon"` string, the coordinate is sent
directly to **Photon** (`/reverse`) without consulting Nominatim. Photon returns a GeoJSON
feature whose `properties` include name, street, city, and country fields.

Photon is backed by a pre-built Elasticsearch index derived from OpenStreetMap data.


In [ ]:
# Reverse geocode Parliament Square
result_rev = await geocoder("51.5007, -0.1246", client)
pprint.pprint(result_rev)

In [ ]:
# Display key properties
addr_rev = result_rev.address if result_rev else None
geo_rev = result_rev.geo if result_rev else None

print(
    f"Coordinates  : {geo_rev.lat if geo_rev else '-'}, {geo_rev.lon if geo_rev else '-'}"
)
print(f"Name         : {addr_rev.name or '-' if addr_rev else '-'}")
print(f"Street       : {addr_rev.street or '-' if addr_rev else '-'}")
print(f"City         : {addr_rev.city or '-' if addr_rev else '-'}")
print(f"Country code : {addr_rev.country_code or '-' if addr_rev else '-'}")

In [ ]:
# Map the reverse geocoding result
lat_r = geo_rev.lat if geo_rev else 51.5007
lon_r = geo_rev.lon if geo_rev else -0.1246
name_r = (addr_rev.name if addr_rev else None) or "Parliament Square"

m_rev = folium.Map(location=[lat_r, lon_r], zoom_start=16)
folium.Marker(
    [lat_r, lon_r],
    popup=folium.Popup(
        f"<b>{name_r}</b><br>lat={lat_r:.5f}, lon={lon_r:.5f}", max_width=250
    ),
    tooltip="Reverse geocoding result",
    icon=folium.Icon(color="green", icon="map-marker"),
).add_to(m_rev)
m_rev

______________________________________________________________________

## 3 — Error Handling

The geocoding functions follow a defensive pattern:

- **`photon_reverse_geocode`**: connection error / timeout → returns `{}`
- **`geocoder`**: connection error, timeout, or empty Nominatim results → returns `None`


In [ ]:
from unittest.mock import patch

from airgap_geo.geocoding import photon_reverse_geocode

# Case 1: unreachable service (bad port)
with patch("airgap_geo.geocoding.PHOTON_API", "http://localhost:19999"):
    bad_rev = await photon_reverse_geocode(51.5, -0.1, client)
print(f"Unreachable Photon \u2192 {bad_rev!r}  (expected: {{}})")

# Case 2: empty Nominatim results (nonsense query)
empty_result = await geocoder("xyzzy_this_place_does_not_exist_anywhere_12345", client)
print(f"Unknown place      \u2192 {empty_result!r}  (expected: None)")

______________________________________________________________________

## Teardown — Close HTTP Client


In [ ]:
await client.aclose()

______________________________________________________________________

## Stop Services (optional)

Run the cell below to stop and remove all containers. Persistent data volumes
are **not** removed.


In [ ]:
def stop_services() -> None:
    """Stop the combined Docker Compose stack. Data volumes are preserved."""
    print(f"Stopping stack from {_COMPOSE_FILE.relative_to(_REPO_ROOT)} ...")
    proc = subprocess.run(
        [
            "docker",
            "compose",
            "-f",
            str(_COMPOSE_FILE),
            "--env-file",
            str(_ENV_FILE),
            "down",
        ],
        capture_output=True,
        text=True,
    )
    if proc.stdout.strip():
        print(proc.stdout.strip())
    if proc.stderr.strip():
        print(proc.stderr.strip())
    print("Stack stopped.")


stop_services()